# Phase-3 train launcher

Runs the frozen `Phase3_Train.ipynb` (SHA-256 `5d0c20e318c8…`) without opening it, because its embedded package freezes the Colab editor (D057).

Before launch, put both files in `MyDrive/Hinglish_Phase3/` (upload through drive.google.com; **do not open** the train notebook):
- `Phase3_Train.ipynb` (unchanged)
- `IMPLEMENTATION_MANIFEST.md`

Runtime: T4 GPU, fresh. Run cell 1, then cell 2. One training run per session.

In [ ]:
import hashlib, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
_STORE = Path('/content/drive/MyDrive/Hinglish_Phase3')
_NB = _STORE / 'Phase3_Train.ipynb'
assert _NB.is_file(), f'Upload the unchanged Phase3_Train.ipynb to {_NB} (do not open it).'
_raw = _NB.read_bytes()
assert hashlib.sha256(_raw).hexdigest() == '5d0c20e318c89a44871c20b2220e044cb6fdbb1ef6b82d8731dff244a13afec4', 'Phase3_Train.ipynb on Drive is not the frozen file; re-upload it unchanged.'
_manifest = _STORE / 'IMPLEMENTATION_MANIFEST.md'
assert _manifest.is_file() and '5d0c20e318c89a44871c20b2220e044cb6fdbb1ef6b82d8731dff244a13afec4' in _manifest.read_text(), 'Manifest missing or does not list the frozen train notebook.'
_cells = [''.join(c['source']) for c in json.loads(_raw)['cells'] if c['cell_type'] == 'code']
assert len(_cells) == 2, len(_cells)

def _sub(src, old, new):
    assert src.count(old) == 1, old
    return src.replace(old, new)

_src = _sub(_cells[0], "EXPECTED_IMPLEMENTATION_SHA256 = ''\n",
            "EXPECTED_IMPLEMENTATION_SHA256 = '824671917d4782a01f458aed001c56671869eae6b494bf837568d2f83c7184df'\n")
print('Frozen notebook verified; running its setup cell (Drive, package checks, pip, GPU checks)...')
get_ipython().run_cell(_src).raise_error()
print('Setup cell finished.')

In [ ]:
# Leave both False unless Claude tells you otherwise after an interrupted run.
RECOVER_STALE_WRITER_OVERRIDE = False
RECOVER_STALE_LOCK_OVERRIDE = False
_src2 = _cells[1]
if RECOVER_STALE_WRITER_OVERRIDE:
    _src2 = _sub(_src2, 'RECOVER_STALE_WRITER = False', 'RECOVER_STALE_WRITER = True')
if RECOVER_STALE_LOCK_OVERRIDE:
    _src2 = _sub(_src2, 'RECOVER_STALE_LOCK = False', 'RECOVER_STALE_LOCK = True')
get_ipython().run_cell(_src2).raise_error()